# Enhancing RoomFormer with DinoV3

## 1. Overall

RoomFormer: from the paper [Connecting the Dots: Floorplan Reconstruction Using Two-Level Queries](https://github.com/ywyue/RoomFormer/tree/main#preparation)  
DINOv3: from [DINOv3](https://github.com/facebookresearch/dinov3/tree/main)  
- We want to utilize the overall architecture of RoomFormer and further improving the results on floorplan reconstruction
- In RoomFormer, 3D point cloud are projected to a density map before going to the rest of the model
- Our approach:
    - Remove the projection phase, directly feeding the point cloud to the model by leveraging DINOv3 as an intermediate layer
    - Replace current encoder with [LitePT](https://github.com/prs-eth/LitePT), keeping decoder and two-level queries intact. 

## 2. Current progress

Note: it is hard to take a large step (i.e. directly adding a 3D point cloud preprocessing layer to RoomFormer), so the work is divided to smaller steps.

### 2.1 Retraining RoomFormer

We want to reproduce the result reported in the paper of RoomFormer. The actual retrained results are shown below:

|                              | room_prec ↑ | room_rec ↑ | corner_prec ↑ | corner_rec ↑ | angles_prec ↑ | angles_rec ↑ | room_f1 ↑ | corner_f1 ↑ | angles_f1 ↑ |
|------------------------------|-------------|------------|---------------|--------------|---------------|--------------|-----------|-------------|-------------|
Results in paper               |        97.9 |       96.7 |          89.1 |         85.3 |          83.0 |         79.5 |      97.3 |        87.2 |        81.2 |
Provided model                 |        97.9 |       96.8 |          89.2 |         85.3 |          83.0 |         79.4 |      97.4 |        87.3 |        81.2 |
Retrained model                |        89.2 |       87.0 |         77.08 |         72.0 |          68.0 |         63.5 |      88.0 |        74.5 |        65.7 |
Retrained model (~1000 epochs) |        91.5 |       89.9 |          79.9 |         76.7 |          72.2 |         69.3 |      90.7 |        78.2 |        70.7 |

As can be seen, the retrained results are nowhere near the reported ones. Currently waiting for response from the author on checking if there is anything settings/hyperparameter missing.

### 2.2 Adding DINOv3 feaures on top of density map

While waiting for the reply from the author(s), it's best we move on to the next step.  

DINOv3 is involved here as an enrichment layer for the density map input of RoomFormer. Specifically, we passed the density map through RoomFormer to produce a patch token of size $(batch\_size \times embed\_dim \times 16 \times 16)$, further pass it through a linear head (consists of a convolution layer and a batch norm layer) and interpolate to get the final feature map of size $(batch\_size \times 1 \times img\_size \times img\_size)$.

![Linear Head overall architecture](./imgs/modified_rf_v1_linear_head.png "Linear Head overall architecture")

The results are reported in the table below:

|                              | room_prec ↑ | room_rec ↑ | corner_prec ↑ | corner_rec ↑ | angles_prec ↑ | angles_rec ↑ | room_f1 ↑ | corner_f1 ↑ | angles_f1 ↑ |
|------------------------------|-------------|------------|---------------|--------------|---------------|--------------|-----------|-------------|-------------|
Results in paper               |        97.9 |       96.7 |          89.1 |         85.3 |          83.0 |         79.5 |      97.3 |        87.2 |        81.2 |
Provided model                 |        97.9 |       96.8 |          89.2 |         85.3 |          83.0 |         79.4 |      97.4 |        87.3 |        81.2 |
Retrained model                |        89.2 |       87.0 |          77.1 |         72.0 |          68.0 |         63.5 |      88.0 |        74.5 |        65.7 |
Retrained model (~1000 epochs) |        91.5 |       89.9 |          79.9 |         76.7 |          72.2 |         69.3 |      90.7 |        78.2 |        70.7 |
DINOv3-enhanced model (v1)     |        88.2 |       87.2 |          77.1 |         70.5 |          68.2 |         62.5 |      87.7 |        73.7 |        65.2 |
DINOv3-enhanced model (v2)     |        93.0 |       91.8 |          84.1 |         78.6 |          76.2 |         71.3 |      92.4 |        81.2 |        73.7 |

Overall, the results are discouraging. While both the original (retrained) and the modified models perform subpar to the provided model, DINOv3-enhanced model shown little to no improvement over the retrained model.

Running cross evaluation on scenecad for original paper and v2:
|                              | IoU ↑ | corner_prec ↑ | corner_rec ↑ | angles_prec ↑ | angles_rec ↑ | corner_f1 ↑ | angles_f1 ↑ |
|------------------------------|-------|---------------|--------------|---------------|--------------|-------------|-------------|
Results in paper               |  74.0 |          56.2 |         65.0 |          44.2 |         48.4 |        60.3 |        46.2 |
Provided model                 |  77.2 |          49.6 |         72.5 |          37.3 |         51.9 |        58.9 |        43.4 |
Retrained model (~1000 epochs) |  51.8 |          40.2 |         45.0 |          30.7 |         34.3 |        42.4 |        32.4 |
DINOv3-enhanced model (v2)     |  72.4 |          66.1 |         65.4 |          53.1 |         52.9 |        65.7 |        53.0 |


In [1]:
import numpy as np

class ArgsTmp:
    def __init__(self):
        pass

args = ArgsTmp

args.lr = 2e-4
args.lr_backbone_names = ['backbone.0']
args.lr_backbone = 2e-5
args.lr_linear_proj_names = ['sampling_offsets']
args.lr_linear_proj_mult = 0.1
args.batch_size = 10
args.weight_decay = 1e-4
args.epochs = 500
args.lr_drop = [400]
args.clip_max_norm = 0.1

args.sgd = False

# backbone
args.backbone = 'resnet50'
args.dilation = False
args.position_embedding = 'sine'
args.position_embedding_scale = 2 * np.pi
args.num_feature_levels = 4

# Transformer
args.enc_layers = 6
args.dec_layers = 6
args.dim_feedforward = 1024
args.hidden_dim = 256
args.dropout = 0.1
args.nheads = 8 #, type=int,
                # help="Number of attention heads inside the transformer's attentions")
args.num_queries = 800 #, type=int,
                # help="Number of query slots (num_polys * max. number of corner per poly)")
args.num_polys = 20 #, type=int,
                # help="Number of maximum number of room polygons")
args.dec_n_points = 4 #, type=int)
args.enc_n_points = 4 #, type=int)
args.query_pos_type = 'sine' #, type=str, choices=('static', 'sine', 'none'),
                # help="Type of query pos in decoder - \
                    # 1. static: same setting with DETR and Deformable-DETR, the query_pos is the same for all layers \
                    # 2. sine: since embedding from reference points (so if references points update, query_pos also \
                    # 3. none: remove query_pos")
args.with_poly_refine = True #, action='store_true',
                # help="iteratively refine reference points (i.e. positional part of polygon queries)")
args.masked_attn = False #, action='store_true',
                # help="if true, the query in one room will not be allowed to attend other room")
args.semantic_classes = -1 #, type=int,
                # help="Number of classes for semantically-rich floorplan:  \
                    # 1. default -1 means non-semantic floorplan \
                    # 2. 19 for Structured3D: 16 room types + 1 door + 1 window + 1 empty")

# loss
args.aux_loss = False #', dest='aux_loss', action='store_true',
                # help="Disables auxiliary decoding losses (loss at each layer)")

# matcher
args.set_cost_class = 2 #, type=float,
                # help="Class coefficient in the matching cost")
args.set_cost_coords = 5 #, type=float,
                # help="L1 coords coefficient in the matching cost")

# loss coefficients
args.cls_loss_coef = 2 #, type=float)
args.room_cls_loss_coef = 0.2 #, type=float)
args.coords_loss_coef = 5 #, type=float)
args.raster_loss_coef = 1 #, type=float)

# dataset parameters
args.dataset_name = 'stru3d'
args.dataset_root = 'data/stru3d' #, type=str)

args.output_dir = 'output',
                # help='path where to save, empty for no saving')
args.device = 'cuda',
                # help='device to use for training / testing')
args.seed = 42 #, type=int)
args.resume = '',# help='resume from checkpoint')
args.start_epoch = 0 #, type=int, metavar='N',
                # help='start epoch')
args.num_workers = 2 #, type=int)
args.job_name = 'train_stru3d' #, type=str)

args.wandb = False #, action='store_true',# help='if added, initiate remote logging')

args.dinov3_repo = 'dinov3'
args.dinov3_checkpoint = 'checkpoints/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
args.dinov3_n_last_layers = 4
args.lr_dinov3_head = 1e-3

args.device = 'cuda'


In [2]:
import torch
# from torch import 

DEVICE = 'cuda'

dinov3 = torch.hub.load(
    args.dinov3_repo,
    "dinov3_vits16",
    source="local",
    weights=args.dinov3_checkpoint,
).to(DEVICE)

dinov3

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (rope_embed): RopePositionEmbedding()
  (blocks): ModuleList(
    (0-11): 12 x SelfAttentionBlock(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (attn): SelfAttention(
        (qkv): LinearKMaskedBias(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
    

In [5]:
from datasets import build_dataset

dataset_train = build_dataset(image_set='train', args=args)

loading annotations into memory...
Done (t=0.24s)
creating index...
index created!
